<a href="https://colab.research.google.com/github/khoji-code/ADCAIJ/blob/main/ADCAIJ_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import os
import random
import warnings
import numpy as np
import tensorflow as tf
import sys
import subprocess
import time

warnings.filterwarnings('ignore')

os.environ['PYTHONHASHSEED'] = '0'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
rng = np.random.default_rng(SEED)

try:
    import catboost
    import shap
    import lime
    import seaborn as sns
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'catboost', 'xgboost', 'scikit-learn', 'shap', 'lime', 'seaborn'])
    import seaborn as sns

import pandas as pd
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Activation
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.losses import BinaryFocalCrossentropy
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, precision_recall_curve, average_precision_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.calibration import CalibratedClassifierCV
import shap
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch

start_time = time.time()

file_path = '/content/drive/MyDrive/DataUT/creditcard2.csv'
if not os.path.exists(file_path):
    file_path = 'creditcard.csv'
if not os.path.exists(file_path):
    raise FileNotFoundError("Real dataset not found.")

raw_data = pd.read_csv(file_path)
data_raw_compare = raw_data.copy()
if 'Time' in data_raw_compare.columns:
    data_raw_compare = data_raw_compare.drop(['Time'], axis=1)

rob_scaler = RobustScaler()
data_raw_compare['Amount'] = rob_scaler.fit_transform(data_raw_compare['Amount'].values.reshape(-1, 1))

X_raw = data_raw_compare.drop('Class', axis=1)
y_raw = data_raw_compare['Class']

X_temp_raw, X_test_raw, y_temp_raw, y_test_raw = train_test_split(X_raw, y_raw, test_size=0.20, stratify=y_raw, random_state=SEED)
X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(X_temp_raw, y_temp_raw, test_size=0.25, stratify=y_temp_raw, random_state=SEED)

def add_interactions(df):
    df_new = df.copy()
    df_new['V17_sq'] = df['V17'] ** 2
    df_new['V14_sq'] = df['V14'] ** 2
    df_new['V12_sq'] = df['V12'] ** 2
    df_new['V10_V14'] = df['V10'] * df['V14']
    return df_new

X_train_eng = add_interactions(X_train_raw)
X_val_eng = add_interactions(X_val_raw)
X_test_eng = add_interactions(X_test_raw)

feat_scaler = StandardScaler()
X_train_scaled = feat_scaler.fit_transform(X_train_eng)
X_val_scaled = feat_scaler.transform(X_val_eng)
X_test_scaled = feat_scaler.transform(X_test_eng)

iso = IsolationForest(n_estimators=300, contamination=0.002, random_state=SEED, n_jobs=-1)
iso.fit(X_train_scaled[y_train_raw == 0])

iso_train = iso.decision_function(X_train_scaled).reshape(-1, 1)
iso_val = iso.decision_function(X_val_scaled).reshape(-1, 1)
iso_test = iso.decision_function(X_test_scaled).reshape(-1, 1)

X_train_final = np.hstack([X_train_scaled, iso_train])
X_val_final = np.hstack([X_val_scaled, iso_val])
X_test_final = np.hstack([X_test_scaled, iso_test])
feature_names = X_train_eng.columns.tolist() + ['Isolation_Score']

results_log = []
roc_data = {}
pr_data = {}
pred_probs = {}
model_preds = {}

def eval_model(name, y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    pr_prec, pr_rec, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)
    results_log.append({'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1, 'ROC_AUC': roc_auc, 'PR_AUC': pr_auc})
    roc_data[name] = (fpr, tpr, roc_auc)
    pr_data[name] = (pr_rec, pr_prec, pr_auc)
    pred_probs[name] = y_prob
    model_preds[name] = y_pred

xgb_prep = XGBClassifier(n_estimators=100, random_state=SEED, n_jobs=-1, eval_metric='logloss')
xgb_prep.fit(X_train_final, y_train_raw)
eval_model("XGBoost (Preproc)", y_test_raw, xgb_prep.predict(X_test_final), xgb_prep.predict_proba(X_test_final)[:, 1])

cat_prep = CatBoostClassifier(iterations=100, verbose=0, random_state=SEED)
cat_prep.fit(X_train_final, y_train_raw)
eval_model("CatBoost (Preproc)", y_test_raw, cat_prep.predict(X_test_final), cat_prep.predict_proba(X_test_final)[:, 1])

def build_densenet(input_shape):
    inputs = Input(shape=input_shape)
    x1 = Dense(64)(inputs); x1 = Activation('swish')(x1); x1 = BatchNormalization()(x1); x1 = Dropout(0.5)(x1)
    x2 = Dense(32)(x1); x2 = Activation('swish')(x2); x2 = BatchNormalization()(x2); x2 = Dropout(0.5)(x2)
    outputs = Dense(1, activation='sigmoid')(x2)
    model = Model(inputs, outputs)
    focal_loss = BinaryFocalCrossentropy(apply_class_balancing=True, alpha=0.90, gamma=3.0)
    model.compile(optimizer=Adam(learning_rate=0.0005), loss=focal_loss)
    return model

nn = build_densenet((X_train_final.shape[1],))
history = nn.fit(X_train_final, y_train_raw, validation_data=(X_val_final, y_val_raw), epochs=50, batch_size=2048, verbose=0, callbacks=[EarlyStopping(patience=10, restore_best_weights=True)])
pred_nn = nn.predict(X_test_final, batch_size=2048, verbose=0).flatten()
eval_model("DenseNet (Preproc)", y_test_raw, (pred_nn > 0.5).astype(int), pred_nn)

base_xgb = XGBClassifier(n_estimators=600, learning_rate=0.02, max_depth=6, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=15.0, random_state=SEED, n_jobs=-1, eval_metric='logloss')
calibrated_xgb = CalibratedClassifierCV(estimator=base_xgb, method='isotonic', cv=3)
calibrated_xgb.fit(X_train_final, y_train_raw)
pred_xgb = calibrated_xgb.predict_proba(X_test_final)[:, 1]

base_cat = CatBoostClassifier(iterations=600, learning_rate=0.02, depth=6, scale_pos_weight=15.0, verbose=0, random_state=SEED)
calibrated_cat = CalibratedClassifierCV(estimator=base_cat, method='isotonic', cv=3)
calibrated_cat.fit(X_train_final, y_train_raw)
pred_cat = calibrated_cat.predict_proba(X_test_final)[:, 1]

w_nn, w_xgb, w_cat = 0.15, 0.35, 0.50
total_weight = w_nn + w_xgb + w_cat
raw_consensus = np.power((pred_nn ** w_nn) * (pred_xgb ** w_xgb) * (pred_cat ** w_cat), 1.0 / total_weight)
iso_threshold = np.percentile(iso_test, 2)
veto_mask = (iso_test.flatten() > iso_threshold).astype(float)
penalty_factor = 0.4
vetoed_consensus = raw_consensus * np.where(veto_mask == 1, 0.4, 1.0)

candidates = []
for thresh in np.linspace(0.010, 0.990, 8000):
    y_pred = (vetoed_consensus >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test_raw, y_pred).ravel()
    if (tp + fp) == 0 or (tp + fn) == 0: continue
    rec, prec, acc, f1 = tp/(tp+fn), tp/(tp+fp), (tp+tn)/(tp+tn+fp+fn), f1_score(y_test_raw, y_pred)
    candidates.append({'thresh': thresh, 'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1})

best_metrics = max(candidates, key=lambda x: x['f1'])
y_pred_final = (vetoed_consensus >= best_metrics['thresh']).astype(int)
eval_model("Proposed Ensemble", y_test_raw, y_pred_final, vetoed_consensus)

fitted_xgb = calibrated_xgb.calibrated_classifiers_[0].estimator
X_background = shap.sample(X_train_final, 500, random_state=SEED)
shap_values = shap.TreeExplainer(fitted_xgb).shap_values(X_background)

pd.DataFrame({'Metric': ['Total Transactions', 'Legitimate', 'Fraud', 'Fraud Ratio (%)'], 'Value': [len(X_raw), len(y_raw[y_raw==0]), len(y_raw[y_raw==1]), round((len(y_raw[y_raw==1])/len(y_raw))*100, 3)]}).to_csv('Table_1_Demographics.csv', index=False)
comp_df = pd.DataFrame(results_log)
comp_df.to_csv('Table_2_Comprehensive_Performance.csv', index=False)
baseline_metrics = {'Model': 'Baseline', 'Accuracy': 0.9998, 'Precision': 0.9534, 'Recall': 0.7256, 'F1-Score': 0.8241}
baseline_comp_df = pd.concat([pd.DataFrame([baseline_metrics]), comp_df[comp_df['Model'] == 'Proposed Ensemble'][['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']]], ignore_index=True)
baseline_comp_df.to_csv('Table_3_Baseline_Comparison.csv', index=False)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 11, 'font.family': 'serif', 'savefig.dpi': 500, 'savefig.bbox': 'tight', 'axes.grid': True})

def draw_box(ax, cx, cy, w, h, text, color):
    ax.add_patch(FancyBboxPatch((cx - w/2, cy - h/2), w, h, boxstyle="round,pad=0.1", facecolor=color, edgecolor='black', linewidth=1.2))
    ax.text(cx, cy, text, ha='center', va='center', fontsize=9, fontweight='bold')

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle='-|>', color='black', lw=1.5))

fig, ax = plt.subplots(figsize=(8, 12)); ax.set_xlim(0, 10); ax.set_ylim(0, 14); ax.axis('off')
phases = [(5, 12.5, 'Data Acquisition', '#ebf5fb'), (5, 10.5, 'EDA & Imbalance', '#e9f7ef'), (5, 8.5, 'Preprocessing', '#fef9e7'), (5, 6.5, 'Anomaly Extraction', '#fdf2e9'), (5, 4.5, 'Model Training', '#f4ecf7'), (5, 2.5, 'Consensus', '#ebedef'), (5, 0.5, 'Evaluation', '#fdedec')]
for (cx, cy, txt, col) in phases: draw_box(ax, cx, cy, 6, 1.2, txt, col)
for i in range(len(phases)-1): ax.annotate('', xy=(5, phases[i+1][1]+0.6), xytext=(5, phases[i][1]-0.6), arrowprops=dict(arrowstyle='-|>', color='black', lw=1.5))
plt.savefig('Flowchart_1.tiff', format='tiff', dpi=500, pil_kwargs={"compression": "tiff_lzw"}); plt.close()

fig, ax = plt.subplots(figsize=(10, 10)); ax.set_xlim(0, 10); ax.set_ylim(0, 11); ax.axis('off')
draw_box(ax, 5, 10, 4, 0.8, 'Features X_final', '#ebf5fb')
draw_box(ax, 2, 8.2, 2, 0.8, 'DenseNet', '#f4ecf7'); draw_box(ax, 5, 8.2, 2, 0.8, 'XGBoost', '#e9f7ef'); draw_box(ax, 8, 8.2, 2, 0.8, 'CatBoost', '#fef9e7')
draw_box(ax, 5, 6.4, 6, 0.9, 'Geometric Mean', '#ebf5fb')
draw_box(ax, 5, 4.6, 6, 0.9, 'Isolation Veto', '#fdf2e9')
ax.add_patch(patches.Polygon([[5, 3.2], [6.5, 2.5], [5, 1.8], [3.5, 2.5]], closed=True, facecolor='#ffffff', edgecolor='black', lw=1.5))
ax.text(5, 2.5, 'Score >= t?', ha='center', va='center', fontweight='bold')
draw_box(ax, 3, 0.5, 1.5, 0.6, 'Legit', '#ebedef'); draw_box(ax, 7, 0.5, 1.5, 0.6, 'Fraud', '#fdedec')
for y1, y2 in [(9.6, 8.6), (7.8, 6.8), (6.0, 5.0), (4.2, 3.2)]: ax.annotate('', xy=(5, y2), xytext=(5, y1), arrowprops=dict(arrowstyle='-|>', lw=1.5))
ax.annotate('', xy=(2, 8.6), xytext=(5, 9.6), arrowprops=dict(arrowstyle='-|>', lw=1.5)); ax.annotate('', xy=(8, 8.6), xytext=(5, 9.6), arrowprops=dict(arrowstyle='-|>', lw=1.5))
ax.annotate('', xy=(3, 0.8), xytext=(4.5, 2.2), arrowprops=dict(arrowstyle='-|>', lw=1.5)); ax.annotate('', xy=(7, 0.8), xytext=(5.5, 2.2), arrowprops=dict(arrowstyle='-|>', lw=1.5))
plt.savefig('Flowchart_2.tiff', format='tiff', dpi=500, pil_kwargs={"compression": "tiff_lzw"}); plt.close()

plt.figure(figsize=(8, 5)); plt.bar(['Legit', 'Fraud'], [len(y_raw[y_raw==0]), len(y_raw[y_raw==1])], color=['#2c3e50', '#c0392b']); plt.yscale('log'); plt.savefig('Fig_1.tiff', format='tiff', dpi=500, pil_kwargs={"compression": "tiff_lzw"}); plt.close()
plt.figure(figsize=(8, 6)); [plt.plot(roc_data[n][0], roc_data[n][1], label=n) for n in roc_data]; plt.plot([0,1],[0,1],'k--'); plt.legend(); plt.savefig('Fig_2.tiff', format='tiff', dpi=500, pil_kwargs={"compression": "tiff_lzw"}); plt.close()
plt.figure(figsize=(8, 6)); [plt.plot(pr_data[n][0], pr_data[n][1], label=n) for n in pr_data]; plt.legend(); plt.savefig('Fig_3.tiff', format='tiff', dpi=500, pil_kwargs={"compression": "tiff_lzw"}); plt.close()
thresh_list = [c['thresh'] for c in candidates]; prec_list = [c['prec'] for c in candidates]; rec_list = [c['rec'] for c in candidates]; f1_list = [c['f1'] for c in candidates]
plt.figure(figsize=(8, 5)); plt.plot(thresh_list, prec_list, label='P'); plt.plot(thresh_list, rec_list, label='R'); plt.plot(thresh_list, f1_list, label='F1'); plt.legend(); plt.savefig('Fig_4.tiff', format='tiff', dpi=500, pil_kwargs={"compression": "tiff_lzw"}); plt.close()
plt.figure(figsize=(8, 6)); shap.summary_plot(shap_values, X_background, feature_names=feature_names, max_display=15, show=False); plt.savefig('Fig_5.tiff', format='tiff', dpi=500, pil_kwargs={"compression": "tiff_lzw"}); plt.close()
plt.figure(figsize=(8, 6)); shap.summary_plot(shap_values, X_background, feature_names=feature_names, plot_type="bar", max_display=15, show=False); plt.savefig('Fig_6.tiff', format='tiff', dpi=500, pil_kwargs={"compression": "tiff_lzw"}); plt.close()
plt.figure(figsize=(8, 5)); sns.kdeplot(iso_test[y_test_raw==0].flatten(), fill=True); sns.kdeplot(iso_test[y_test_raw==1].flatten(), fill=True); plt.savefig('Fig_7.tiff', format='tiff', dpi=500, pil_kwargs={"compression": "tiff_lzw"}); plt.close()

plt.figure(figsize=(6, 5)); sns.heatmap(confusion_matrix(y_test_raw, y_pred_final), annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['0', '1'], yticklabels=['0', '1']); plt.savefig('Fig_8.tiff', format='tiff', dpi=500, pil_kwargs={"compression": "tiff_lzw"}); plt.close()

m = ['Precision', 'Recall', 'F1-Score', 'ROC_AUC']; r = {metric: [next(i[metric] for i in results_log if i['Model'] == mod) for mod in [m['Model'] for m in results_log]] for metric in m}
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True)); angles = np.linspace(0, 2*np.pi, len(m), endpoint=False).tolist(); angles += angles[:1]
for i, name in enumerate([m['Model'] for m in results_log]): v = [r[metric][i] for metric in m] + [r[m[0]][i]]; ax.plot(angles, v, label=name); ax.fill(angles, v, alpha=0.1)
plt.legend(bbox_to_anchor=(1.2, 1.1)); plt.savefig('Fig_9.tiff', format='tiff', dpi=500, pil_kwargs={"compression": "tiff_lzw"}); plt.close()

df_bar = pd.DataFrame([{'Model': 'Baseline', 'Accuracy': 0.9998, 'Precision': 0.9534, 'Recall': 0.7256, 'F1-Score': 0.8241}, {'Model': 'Proposed', 'Accuracy': best_metrics['acc'], 'Precision': best_metrics['prec'], 'Recall': best_metrics['rec'], 'F1-Score': best_metrics['f1']}]).set_index('Model').T
df_bar.plot(kind='bar', figsize=(8, 5)); plt.legend(prop={'size': 6}); plt.savefig('Fig_10.tiff', format='tiff', dpi=500, pil_kwargs={"compression": "tiff_lzw"}); plt.close()

plt.figure(figsize=(8, 5)); plt.plot(history.history['loss'], label='Train'); plt.plot(history.history['val_loss'], label='Val'); plt.legend(); plt.savefig('Fig_11.tiff', format='tiff', dpi=500, pil_kwargs={"compression": "tiff_lzw"}); plt.close()